# Wiki300 冻结v1历史核查（不是现行审核标签）

只读缓存、重评分及人工待审状态；不训练真实模型、不改标签、不联网。结构通过不等于人类Gold。完整审计代码在scripts/audit_dataset.py。


In [1]:
from pathlib import Path
import sys, json, tempfile, contextlib, io
ROOT=Path.cwd().resolve()
while not (ROOT/'artifacts/exp05').is_dir() and ROOT != ROOT.parent:
    ROOT=ROOT.parent
assert (ROOT/'artifacts/exp05').is_dir(), 'Run from the extracted milestone directory'
sys.path.insert(0,str(ROOT/'scripts'))
from audit_dataset import run, csvrows
print('Source: frozen Wiki300; execution does not fit any model')


Source: frozen Wiki300; execution does not fit any model


In [2]:
with tempfile.TemporaryDirectory() as d, contextlib.redirect_stdout(io.StringIO()):
    audit=run(ROOT/'artifacts/exp05', Path(d))
assert audit['status']=='PASS'
print(json.dumps({k:audit[k] for k in ['row_count','label_counts','split_counts','exact_subtypes','positive_normal_prefix_groups','interval_width_groups','control_window_groups','structural_check_n','structural_checks_passed','errors']},ensure_ascii=False,indent=2))


{
  "row_count": 300,
  "label_counts": {
    "EXACT": 61,
    "INTERVAL": 139,
    "NOT_FRAUD": 100
  },
  "split_counts": {
    "TRAIN": 226,
    "VAL": 24,
    "TEST": 50
  },
  "exact_subtypes": {
    "EXACT_PUBLIC": 51,
    "EXACT_OTHER_LOG": 10
  },
  "positive_normal_prefix_groups": {
    "PREFIX_1": 118,
    "PREFIX_2": 30,
    "PREFIX_3PLUS": 52
  },
  "interval_width_groups": {
    "gt_30d": 51,
    "le_1d": 56,
    "7_to_30d": 14,
    "1_to_7d": 18
  },
  "control_window_groups": {
    "lt_1d": 52,
    "ge_7d": 28,
    "1_to_lt_7d": 20
  },
  "structural_check_n": 5064,
  "structural_checks_passed": 5064,
  "errors": []
}


## 已封存预测的分流评分

9个EXACT UTC与8个精确发布位置的分母不同。区间命中不是分类准确率；正常窗口没有风险预测，人工Gold仍为0。


In [3]:
scores=json.loads((ROOT/'reports/evaluation_v2/exploratory_score_v2.json').read_text())
print('EXACT UTC n=',scores['exact_point_utc_n'])
print('Mean absolute clock error (hours):', {k:round(v*24,6) for k,v in scores['exact_point_utc_mae_days'].items()})
print('Exact published event n=',scores['exact_published_event_n'])
print('Event-position MAE:',scores['exact_event_abs_error_mean'])
print('Interval strata:',json.dumps(scores['interval_stratified'],ensure_ascii=False,indent=2))


EXACT UTC n= 9
Mean absolute clock error (hours): {'fusion': 21.216512, 'behavior': 18.479969, 'delay': 85.133241}
Exact published event n= 8
Event-position MAE: {'fusion': 5.875, 'behavior': 5.5, 'delay': 5.125}
Interval strata: {
  "1_to_7d": {
    "fusion": {
      "n": 5,
      "hits": 1,
      "hit_rate": 0.2,
      "median_distance_days": 0.033622685185185186
    },
    "behavior": {
      "n": 5,
      "hits": 1,
      "hit_rate": 0.2,
      "median_distance_days": 0.0309375
    },
    "delay": {
      "n": 5,
      "hits": 2,
      "hit_rate": 0.4,
      "median_distance_days": 0.027650462962962963
    }
  },
  "7_to_30d": {
    "fusion": {
      "n": 3,
      "hits": 3,
      "hit_rate": 1.0,
      "median_distance_days": 0.0
    },
    "behavior": {
      "n": 3,
      "hits": 3,
      "hit_rate": 1.0,
      "median_distance_days": 0.0
    },
    "delay": {
      "n": 3,
      "hits": 2,
      "hit_rate": 0.6666666666666666,
      "median_distance_days": 0.0
    }
  },
  "gt_

In [4]:
from collections import Counter
QUEUE_ROOT=ROOT/'history/review_before_rules_v1_1/HUMAN_REVIEW_QUEUE'
if not QUEUE_ROOT.exists(): QUEUE_ROOT=ROOT/'HUMAN_REVIEW_QUEUE'
queue=csvrows(QUEUE_ROOT/'HUMAN_REVIEW_QUEUE.csv')
assert len(queue)==12 and all(x['split']=='TRAIN' for x in queue)
assert all(x['human_gold_status']=='PENDING' for x in queue)
print('Review tasks:',len(queue),'TRAIN:',len(queue),'Human Gold complete:',0)
print('Original source types:',dict(Counter(x['source_label_type'] for x in queue)))
print('Dataset readiness: 8/10 project gates = 80%; engineering deliverables: 10/10 = 100%. These are not accuracy estimates.')


Review tasks: 12 TRAIN: 12 Human Gold complete: 0
Original source types: {'EXACT': 2, 'EXACT_PUBLIC_REVISION': 1, 'EXACT_PUBLIC_LOG_TIMESTAMP': 1, 'INTERVAL': 4, 'RIGHT_CENSORED': 4}
Dataset readiness: 8/10 project gates = 80%; engineering deliverables: 10/10 = 100%. These are not accuracy estimates.
